In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\Raw_data_1Day_2024_site_1431_Patparganj_Delhi_DPCC_1Day.csv")

In [3]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),...,MP-Xylene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s)
0,2024-01-01,186.73,336.17,7.55,41.91,28.43,29.94,1.83,0.75,15.96,...,NaN,10.16,78.08,1.72,276.68,0.00,0.00,70.79,994.93,NaN
1,2024-01-02,193.67,346.95,11.22,44.13,32.64,27.37,3.32,0.80,14.15,...,NaN,10.00,73.57,1.62,250.54,0.00,0.00,87.15,994.26,NaN
2,2024-01-03,222.98,402.27,24.36,46.28,44.45,31.42,3.32,1.26,20.15,...,NaN,9.49,85.53,1.86,200.31,0.00,0.00,77.89,994.13,NaN
3,2024-01-04,252.04,453.63,25.03,44.77,44.21,33.31,2.70,1.04,16.03,...,NaN,10.02,84.60,1.33,304.96,0.00,0.00,70.14,994.36,NaN
4,2024-01-05,188.85,339.91,12.82,41.51,32.62,38.07,5.86,0.98,13.05,...,NaN,10.64,87.33,1.49,319.92,0.00,0.00,64.44,994.36,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,187.62,247.17,13.52,53.07,39.24,47.19,0.69,1.40,10.87,...,NaN,15.20,83.21,2.89,9.91,1.14,1.14,27.59,990.47,NaN
362,2024-12-28,105.88,144.21,28.65,46.16,47.85,39.48,2.20,1.67,4.24,...,NaN,15.12,84.61,1.30,9.76,0.06,0.06,27.73,987.10,NaN
363,2024-12-29,115.58,167.29,5.76,27.12,19.12,30.27,1.06,1.33,17.83,...,NaN,14.90,82.28,2.35,23.26,0.00,0.00,71.92,987.38,NaN
364,2024-12-30,108.42,166.33,5.16,29.52,19.91,16.25,0.72,1.03,25.84,...,NaN,13.55,78.07,1.92,23.92,0.00,0.00,74.27,987.10,NaN


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (366, 21)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['Xylene (µg/m³)']
Dropped rows (>70% NaN): 1
Missing values after imputation:
 Timestamp          0
PM2.5 (µg/m³)      0
PM10 (µg/m³)       0
NO (µg/m³)         0
NO2 (µg/m³)        0
NOx (ppb)          0
NH3 (µg/m³)        0
SO2 (µg/m³)        0
CO (mg/m³)         0
Ozone (µg/m³)      0
Benzene (µg/m³)    0
Toluene (µg/m³)    0
AT (°C)            0
RH (%)             0
WS (m/s)           0
WD (deg)           0
RF (mm)            0
TOT-RF (mm)        0
SR (W/mt2)         0
BP (mmHg)          0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (365, 20)
    Timestamp  PM2.5 (µg/m³)  PM10 (µg/m³)  NO (µg/m³)  NO2 (µg/m³)  \
0  2024-01-01         186.73        336.17        7.55        41.91   
1  2024-01-02         193.67        346.95       11.22        44.13   
2  2024-01-03         222.98        402.27       24.36        46.28   
3  2024-01-04         252.04        453.63       25.03        44.77   
4  2024-01-05         188.85        339.91       12.82        41.51   

   NOx (ppb)  NH3 (µg/m³)  SO2 (µg/m³)  CO (mg/m³)  Ozone (µg/m³)  \
0      28.43        29.94         1.83        0.75          15.96   
1      32.64        27.37         3.32        0.80          14.15   
2      44.45        31.42         3.32        1.26          20.15   
3      44.21        33.31         2.70        1.04          16.03   
4      32.62        38.07         5.86        0.98          13.05   

   Benzene (µg/m³)  Toluene (µg/m³)  AT (°C)  RH (%)  WS (m/s)  WD (deg)  \
0             0.54             2.39    10.16   78.08      1

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),Benzene (µg/m³),Toluene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg)
0,2024-01-01,1.382379,0.790351,-0.519919,0.110765,-0.239438,-0.250339,-1.266043,-0.447329,-0.994679,0.203855,-0.380891,-1.753099,1.157884,0.080139,0.166649,0.0,0.0,-0.804254,2.138396
1,2024-01-02,1.481541,0.876583,-0.217795,0.244564,-0.014936,-0.560728,-0.897607,-0.342437,-1.082664,0.136579,-0.532130,-1.770986,0.877763,-0.048776,0.166649,0.0,0.0,-0.478952,2.025284
2,2024-01-03,1.900339,1.319102,0.863927,0.374145,0.614845,-0.071594,-0.897607,0.622571,-0.791000,1.179357,-0.126809,-1.828000,1.620613,0.260621,0.166649,0.0,0.0,-0.663078,2.003337
3,2024-01-04,2.315564,1.729944,0.919083,0.283137,0.602047,0.156668,-1.050916,0.161046,-0.991276,0.607511,0.266413,-1.768750,1.562849,-0.422631,0.166649,0.0,0.0,-0.817178,2.042167
4,2024-01-05,1.412670,0.820268,-0.086078,0.086657,-0.016002,0.731552,-0.269534,0.035175,-1.136135,0.439321,-0.175205,-1.699438,1.732413,-0.216366,0.166649,0.0,0.0,-0.930517,2.042167
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
360,2024-12-27,1.395096,0.078417,-0.028452,0.783378,0.337016,1.833008,-1.547934,0.916270,-1.242106,0.271131,0.042579,-1.189662,1.476515,1.588450,-1.886093,0.0,0.0,-1.663241,1.385442
361,2024-12-28,0.227149,-0.745187,1.217092,0.366912,0.796154,0.901843,-1.174552,1.482688,-1.564395,-0.031610,0.205917,-1.198605,1.563470,-0.461305,-1.896309,0.0,0.0,-1.660458,0.816506
362,2024-12-29,0.365748,-0.560564,-0.667277,-0.780629,-0.735904,-0.210484,-1.456443,0.769421,-0.903777,-0.536180,-0.883004,-1.223200,1.418751,0.892306,-0.976866,0.0,0.0,-0.781785,0.863777
363,2024-12-30,0.263442,-0.568244,-0.716671,-0.635981,-0.693777,-1.903732,-1.540515,0.140067,-0.514406,-0.569818,-0.737815,-1.374120,1.157263,0.337970,-0.931915,0.0,0.0,-0.735058,0.816506


In [10]:
df.to_excel("patparganj2024.xlsx", index=False)